# <font color="#FFA500">VICTIMAS DIGITALES: ANÁLISIS DE LOS DELITOS INFORMÁTICOS EN COLOMBIA</font>

##1.1 Cargar librerias y bases

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime
import seaborn as sns
from scipy.stats import chi2_contingency
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.tsa.stattools import grangercausalitytests
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

In [2]:
#nueva base filtrada de 30M
#https://drive.google.com/file/d/1coDof8AiFvddDpMyjDN0N7uOl_FggX7y/view?usp=sharing

file_id = '1coDof8AiFvddDpMyjDN0N7uOl_FggX7y'
url = f'https://drive.google.com/uc?id={file_id}'
di = pd.read_csv(url)
di.head()

,Unnamed: 0,CRIMINALIDAD,ES_ARCHIVO,ES_PRECLUSIÓN,ESTADO,ETAPA_CASO,LEY,PAÍS_HECHO,DEPARTAMENTO_HECHO,MUNICIPIO_HECHO,...,GRUPO_DELITO,VÍCTIMA_CONSUMADO,SEXO,GRUPO_ETARIO,PAÍS_NACIMIENTO,APLICA_LGBTI,APLICA_NNA,INDÍGENA,AFRODESCENDIENTE,TOTAL_VÍCTIMAS
0,8,SI,NO,NO,ACTIVO,JUICIO,Ley 906,Colombia,Atlántico,BARRANQUILLA,...,DELITOS INFORMATICOS,NO APLICA,FEMENINO,Adulto entre 27 y 59 años.,Colombia,NO,NO,NO,NO,189
1,20,NO,SI,NO,INACTIVO,INDAGACIÓN,Ley 906,Colombia,Antioquia,BELLO,...,DELITOS INFORMATICOS,NO APLICA,MASCULINO,Adolescente de 14 a 17 años.,SIN DATO,NO,SI,NO,NO,1
2,24,SI,NO,NO,ACTIVO,INDAGACIÓN,Ley 906,Colombia,"BOGOTÁ, D. C.","BOGOTÁ, D.C.",...,DELITOS INFORMATICOS,NO APLICA,MASCULINO,Adulto entre 27 y 59 años.,Colombia,NO,NO,NO,NO,175
3,41,SI,SI,NO,INACTIVO,INDAGACIÓN,Ley 906,Colombia,Córdoba,CERETÉ,...,DELITOS INFORMATICOS,NO APLICA,FEMENINO,Adulto entre 27 y 59 años.,Colombia,NO,NO,NO,NO,2
4,74,SI,SI,NO,INACTIVO,INDAGACIÓN,Ley 906,Colombia,Antioquia,ENVIGADO,...,DELITOS INFORMATICOS,NO APLICA,FEMENINO,Adulto entre 27 y 59 años.,Colombia,NO,NO,NO,NO,36


##1.2 Conocer información básica

In [3]:
# Encontrar número de filas y columnas
di.shape

(105736, 25)

In [4]:
# Conocer información básica del Dataset
di.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105736 entries, 0 to 105735
Data columns (total 25 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   Unnamed: 0          105736 non-null  int64 
 1   CRIMINALIDAD        105736 non-null  object
 2   ES_ARCHIVO          105736 non-null  object
 3   ES_PRECLUSIÓN       105736 non-null  object
 4   ESTADO              105736 non-null  object
 5   ETAPA_CASO          105736 non-null  object
 6   LEY                 105736 non-null  object
 7   PAÍS_HECHO          105736 non-null  object
 8   DEPARTAMENTO_HECHO  105736 non-null  object
 9   MUNICIPIO_HECHO     105736 non-null  object
 10  SECCIONAL           105736 non-null  object
 11  AÑO_HECHOS          105736 non-null  int64 
 12  AÑO_ENTRADA         105736 non-null  object
 13  AÑO_DENUNCIA        105736 non-null  object
 14  DELITO              105736 non-null  object
 15  GRUPO_DELITO        105736 non-null  object
 16  VÍ

In [5]:
# Reconocer los nombres de las columnas
di.columns

Index(['Unnamed: 0', 'CRIMINALIDAD', 'ES_ARCHIVO', 'ES_PRECLUSIÓN', 'ESTADO',
       'ETAPA_CASO', 'LEY', 'PAÍS_HECHO', 'DEPARTAMENTO_HECHO',
       'MUNICIPIO_HECHO', 'SECCIONAL', 'AÑO_HECHOS', 'AÑO_ENTRADA',
       'AÑO_DENUNCIA', 'DELITO', 'GRUPO_DELITO', 'VÍCTIMA_CONSUMADO', 'SEXO',
       'GRUPO_ETARIO', 'PAÍS_NACIMIENTO', 'APLICA_LGBTI', 'APLICA_NNA',
       'INDÍGENA', 'AFRODESCENDIENTE', 'TOTAL_VÍCTIMAS'],
      dtype='object')

In [6]:
# Número total de víctimas
print(di["TOTAL_VÍCTIMAS"].sum())

419907


In [7]:
# Número total de víctimas discrimadas por sexo. Tipo de datos: Numérico
di.groupby("SEXO")["TOTAL_VÍCTIMAS"].sum().sort_values()

,TOTAL_VÍCTIMAS
SEXO,
SIN DATO,23080
MASCULINO,178729
FEMENINO,218098


In [8]:
# Número total de víctimas discriminadas por grupo etario. Tipo de datos: Numérico
di.groupby("GRUPO_ETARIO")["TOTAL_VÍCTIMAS"].sum().sort_values()

,TOTAL_VÍCTIMAS
GRUPO_ETARIO,
"Niño, Niña. Población de 0 a 13 años.",1608
Adolescente de 14 a 17 años.,3285
Joven de 18 a 26 años.,53799
Adulto mayor. Personas igual o mayor a 60 años.,56048
SIN DATO,64280
Adulto entre 27 y 59 años.,240887


In [11]:
# Calcular el total de víctimas para sacar los porcentajes de los SIN DATO
di_total_victimas = di['TOTAL_VÍCTIMAS'].sum()

por_sexo = ( di.groupby("SEXO")["TOTAL_VÍCTIMAS"].sum().sort_values(ascending=False).reset_index())

# Agregar columna de porcentaje
por_sexo['PORCENTAJE'] = (por_sexo['TOTAL_VÍCTIMAS'] / di_total_victimas * 100).round(2)

# Mostrar resultados
por_sexo

,SEXO,TOTAL_VÍCTIMAS,PORCENTAJE
0,FEMENINO,218098,51.94
1,MASCULINO,178729,42.56
2,SIN DATO,23080,5.50


In [12]:
por_edad = ( di.groupby("GRUPO_ETARIO")["TOTAL_VÍCTIMAS"].sum().sort_values(ascending=False).reset_index())

# Agregar columna de porcentaje
por_edad['PORCENTAJE'] = (por_edad['TOTAL_VÍCTIMAS'] / di_total_victimas * 100).round(2)

# Mostrar resultados
por_edad

,GRUPO_ETARIO,TOTAL_VÍCTIMAS,PORCENTAJE
0,Adulto entre 27 y 59 años.,240887,57.37
1,SIN DATO,64280,15.31
2,Adulto mayor. Personas igual o mayor a 60 años.,56048,13.35
3,Joven de 18 a 26 años.,53799,12.81
4,Adolescente de 14 a 17 años.,3285,0.78
5,"Niño, Niña. Población de 0 a 13 años.",1608,0.38


In [13]:
# Generar informe YData
!pip install ydata-profiling

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.1/400.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.5/296.5 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 687.8/687.8 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 93.5 MB/s eta 0:00:00
  Created wheel for htmlmin: filename=htmlmin-0.1.12-py3-none-any.whl size=27081 sha256=0219fb20715c2ffa66c46fcca11ffa1c49f507650040201f61a650c4a6b778f8
  Stored in directory: /root/.cache/pip/wheels/8d/55/1a/19cd535375ed1ede0c996405ebffe34b196d78e2d9545723a2
Successfully built htmlmin


In [14]:
# Generar informe YData
from ydata_profiling import ProfileReport

profile = ProfileReport (di, title= "Reporte de EDA", explorative = True)

profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 25/25 [00:03<00:00,  6.64it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
#
por_dpto = ( di.groupby("DEPARTAMENTO_HECHO")["TOTAL_VÍCTIMAS"].sum().sort_values(ascending=False).reset_index())

# Agregar columna de porcentaje
por_dpto['PORCENTAJE'] = (por_edad['TOTAL_VÍCTIMAS'] / di_total_victimas * 100).round(2)

# Mostrar resultados
por_dpto

,DEPARTAMENTO_HECHO,TOTAL_VÍCTIMAS,PORCENTAJE
0,"BOGOTÁ, D. C.",94991,57.37
1,Antioquia,80207,15.31
2,Valle del Cauca,62129,13.35
3,Atlántico,26649,12.81
4,Cundinamarca,26078,0.78
5,Santander,12539,0.38
6,Tolima,11085,NaN
7,Norte de Santander,10216,NaN
8,Bolívar,9828,NaN
9,Risaralda,8964,NaN


### Calcular medidas de tendencia central de las variables (sexo, edad, departamento)

---

Al tener la gran mayoría de nuestras variables categóricas estaremos calculando la moda como medida de tendencia central de las variables: **SEXO, EDAD, DEPARTAMENTO**

In [19]:
# Moda del sexo donde el sin dato es del 5.5%
di["SEXO"].mode()

,SEXO
0,FEMENINO


In [20]:
#Moda del GRUPO ETARIO donde el sin dato es del 15.3%
di["GRUPO_ETARIO"].mode()

,GRUPO_ETARIO
0,Adulto entre 27 y 59 años.


In [18]:
# Moda del sexo donde el sin dato es del 5.5%
di["DEPARTAMENTO_HECHO"].mode()

,DEPARTAMENTO_HECHO
0,Antioquia


In [24]:
import os
os.getcwd()

'/content'

In [30]:
# prompt: import nbformat
# path = 'content/Fase_filtrada_DI.ipynb'  # Update with the actual filename
# with open(path) as f:
#     nb = nbformat.read(f, as_version=4)
# # Remove widgets metadata
# if 'widgets' in nb['metadata']:
#     del nb['metadata']['widgets']
# with open(path, 'w') as f:
#     nbformat.write(nb, f)
# i need to use it on this notebook

import nbformat
import os

# Specify the correct path to your notebook file.
# Use os.path.join for platform-independent path construction.
path = os.path.join(os.getcwd(), 'Fase_filtrada_DI.ipynb')

# Check if the file exists before attempting to open it
if os.path.exists(path):
    with open(path) as f:
        nb = nbformat.read(f, as_version=4)

    # Remove widgets metadata
    if 'widgets' in nb['metadata']:
        del nb['metadata']['widgets']

    with open(path, 'w') as f:
        nbformat.write(nb, f)

    print(f"Successfully processed {path}")
else:
    print(f"Error: File not found at {path}")


Error: File not found at /content/Fase_filtrada_DI.ipynb


In [28]:
import nbformat

path = 'content/Fase_filtrada_DI.ipynb'  # Update with the actual filename

with open(path) as f:
    nb = nbformat.read(f, as_version=4)

# Remove widgets metadata
if 'widgets' in nb['metadata']:
    del nb['metadata']['widgets']

with open(path, 'w') as f:
    nbformat.write(nb, f)


FileNotFoundError: [Errno 2] No such file or directory: 'content/Fase_filtrada_DI.ipynb'

##1.3 Análisis de Datos EDA


##1.4 Limpieza y Preparación de Datos

##1.5 Análisis Inferencial

##1.6 Análisis Causal